In [1]:

import pandas as pd
import requests
import io
import zipfile

# Download Online Retail II from UCI ML Repository
url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
print("Downloading dataset...")
response = requests.get(url, timeout=120)
print(f"Status: {response.status_code}, Size: {len(response.content)/1e6:.1f} MB")

zf = zipfile.ZipFile(io.BytesIO(response.content))
print("Files in zip:", zf.namelist())

excel_name = [n for n in zf.namelist() if n.endswith('.xlsx')][0]
print(f"Loading: {excel_name}")

with zf.open(excel_name) as f:
    xl = pd.ExcelFile(f)
    print("Sheets:", xl.sheet_names)
    dfs = []
    for sheet in xl.sheet_names:
        _df = pd.read_excel(xl, sheet_name=sheet, dtype={'Invoice': str, 'StockCode': str})
        dfs.append(_df)

raw_df = pd.concat(dfs, ignore_index=True)

# Rename to standard column names immediately so downstream blocks are consistent
raw_df = raw_df.rename(columns={
    'Invoice': 'InvoiceNo',
    'Price': 'UnitPrice',
    'Customer ID': 'CustomerID'
})

print(f"\nRaw shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")
print(f"\nDtypes:\n{raw_df.dtypes}")
print(f"\nFirst 3 rows:\n{raw_df.head(3).to_string()}")
print(f"\nMissing values:\n{raw_df.isnull().sum()}")
print(f"\nQuantity/UnitPrice stats:\n{raw_df[['Quantity','UnitPrice']].describe()}")


Status: 200, Size: 45.6 MB
Files in zip: ['online_retail_II.xlsx']
Loading: online_retail_II.xlsx
Sheets: ['Year 2009-2010', 'Year 2010-2011']

Raw shape: (1067371, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Dtypes:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

First 3 rows:
  InvoiceNo StockCode                          Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country
0    489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00       6.95     13085.0  United Kingdom
1    489434    79323P                   PINK CHERRY LIGHTS        12 2009-12-01 07:45:00       6.75     13085.0  United Kingdom
2    489434    79323W                  WHITE CHERRY LI

In [2]:
import pandas as pd

# raw_df already has standardized column names from Load Data block
# (InvoiceNo, UnitPrice, CustomerID — no rename or _year_sheet column needed)
retail_df = raw_df.copy()

n_raw = len(retail_df)
print(f"Raw rows: {n_raw:,}")

Raw rows: 1,067,371


In [3]:
# 1. Flag and remove cancellations (InvoiceNo starts with 'C')
retail_df['IsCancellation'] = retail_df['InvoiceNo'].astype(str).str.startswith('C')
n_cancel = retail_df['IsCancellation'].sum()
retail_df = retail_df[~retail_df['IsCancellation']].drop(columns=['IsCancellation'])
print(f"Removed cancellations: {n_cancel:,}")

Removed cancellations: 19,494


In [4]:
# 2. Drop rows with missing CustomerID
n_no_cust = retail_df['CustomerID'].isnull().sum()
retail_df = retail_df.dropna(subset=['CustomerID'])
print(f"Removed missing CustomerID: {n_no_cust:,}")

Removed missing CustomerID: 242,257


In [5]:
# 3. Remove non-positive Quantity or UnitPrice
n_bad_qty   = (retail_df['Quantity']   <= 0).sum()
n_bad_price = (retail_df['UnitPrice']  <= 0).sum()
retail_df = retail_df[(retail_df['Quantity'] > 0) & (retail_df['UnitPrice'] > 0)]
print(f"Removed non-positive Quantity: {n_bad_qty:,} | non-positive Price: {n_bad_price:,}")

Removed non-positive Quantity: 0 | non-positive Price: 71


In [6]:
# 4. Deduplicate
n_before_dedup = len(retail_df)
retail_df = retail_df.drop_duplicates()
n_dupes = n_before_dedup - len(retail_df)
print(f"Removed duplicates: {n_dupes:,}")

Removed duplicates: 26,124


In [7]:
# 5. Ensure correct types
retail_df['CustomerID']  = retail_df['CustomerID'].astype(int).astype(str)
retail_df['InvoiceDate'] = pd.to_datetime(retail_df['InvoiceDate'])

In [8]:
# 6. Derived columns
retail_df['Revenue']   = retail_df['Quantity'] * retail_df['UnitPrice']
retail_df['Year']      = retail_df['InvoiceDate'].dt.year
retail_df['Month']     = retail_df['InvoiceDate'].dt.month
retail_df['YearMonth'] = retail_df['InvoiceDate'].dt.to_period('M')

n_clean = len(retail_df)
print(f"\n{'='*45}")
print(f"CLEANED DATASET SUMMARY")
print(f"{'='*45}")
print(f"Raw rows        : {n_raw:>10,}")
print(f"Cleaned rows    : {n_clean:>10,}")
print(f"Rows removed    : {n_raw - n_clean:>10,} ({(n_raw - n_clean) / n_raw * 100:.1f}%)")
print(f"Date range      : {retail_df['InvoiceDate'].min().date()} → {retail_df['InvoiceDate'].max().date()}")
print(f"Unique customers: {retail_df['CustomerID'].nunique():>9,}")
print(f"Unique products : {retail_df['StockCode'].nunique():>9,}")
print(f"Unique invoices : {retail_df['InvoiceNo'].nunique():>9,}")
print(f"Countries       : {retail_df['Country'].nunique():>9,}")
print(f"Total Revenue   : £{retail_df['Revenue'].sum():>12,.2f}")
print(f"\nRevenue stats:\n{retail_df['Revenue'].describe().round(2)}")


CLEANED DATASET SUMMARY
Raw rows        :  1,067,371
Cleaned rows    :    779,425
Rows removed    :    287,946 (27.0%)
Date range      : 2009-12-01 → 2011-12-09
Unique customers:     5,878
Unique products :     4,631
Unique invoices :    36,969
Countries       :        41
Total Revenue   : £17,374,804.27

Revenue stats:
count    779425.00
mean         22.29
std         227.43
min           0.00
25%           4.95
50%          12.48
75%          19.80
max      168469.60
Name: Revenue, dtype: float64


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_revenue   = retail_df['Revenue'].sum()
total_orders    = retail_df['InvoiceNo'].nunique()
total_customers = retail_df['CustomerID'].nunique()
avg_order_value = retail_df.groupby('InvoiceNo')['Revenue'].sum().mean()

print("=== EXECUTIVE KPIs ===")
print(f"Total Revenue      : £{total_revenue:>12,.2f}")
print(f"Total Orders       : {total_orders:>12,}")
print(f"Total Customers    : {total_customers:>12,}")
print(f"Avg Order Value    : £{avg_order_value:>12,.2f}")

=== EXECUTIVE KPIs ===
Total Revenue      : £17,374,804.27
Total Orders       :       36,969
Total Customers    :        5,878
Avg Order Value    : £      469.98


In [10]:
# ── Monthly revenue ───────────────────────────────────────────────────────────
monthly_revenue_df = (
    retail_df.groupby('YearMonth')
    .agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'))
    .reset_index()
    .sort_values('YearMonth')
)
monthly_revenue_df['RevenueK'] = monthly_revenue_df['Revenue'] / 1e3

In [11]:
# ── Seasonal (month-of-year average) ─────────────────────────────────────────
seasonal_df = (
    retail_df.groupby(['Year','Month'])['Revenue'].sum()
    .reset_index()
    .groupby('Month')['Revenue'].mean()
    .reset_index()
)
seasonal_df['MonthName'] = pd.to_datetime(seasonal_df['Month'], format='%m').dt.strftime('%b')

In [12]:
# ── YoY (2010 vs 2011) ───────────────────────────────────────────────────────
yoy_df    = retail_df.groupby(['Year','Month'])['Revenue'].sum().reset_index()
yoy_pivot = yoy_df.pivot(index='Month', columns='Year', values='Revenue').fillna(0)
yoy_pivot.index = pd.to_datetime(yoy_pivot.index, format='%m').strftime('%b')

In [13]:
# ── PLOT 1: Monthly Revenue Trend ─────────────────────────────────────────────
fig_monthly, _ax1 = plt.subplots(figsize=(13, 4.5))
fig_monthly.patch.set_facecolor('#1D1D20')
_ax1.set_facecolor('#1D1D20')
_x = range(len(monthly_revenue_df))
_ax1.fill_between(_x, monthly_revenue_df['RevenueK'], alpha=0.25, color='#A1C9F4')
_ax1.plot(_x, monthly_revenue_df['RevenueK'], color='#A1C9F4', linewidth=2.2, marker='o', markersize=4)
_xt = list(range(0, len(monthly_revenue_df), 3))
_ax1.set_xticks(_xt)
_ax1.set_xticklabels([monthly_revenue_df['YearMonth'].iloc[i] for i in _xt],
                     rotation=45, ha='right', fontsize=9, color='#909094')
_ax1.tick_params(axis='y', colors='#909094', labelsize=9)
_ax1.set_ylabel('Revenue (£000s)', color='#909094', fontsize=10)
_ax1.set_title('Monthly Revenue Trend  (Dec 2009 – Dec 2011)', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax1.spines.values(): _sp.set_edgecolor('#444')
_ax1.grid(axis='y', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [14]:
# ── PLOT 2: Seasonal Average Revenue by Month ─────────────────────────────────
fig_seasonal, _ax2 = plt.subplots(figsize=(9, 4.5))
fig_seasonal.patch.set_facecolor('#1D1D20')
_ax2.set_facecolor('#1D1D20')
_clrs = ['#ffd400' if v == seasonal_df['Revenue'].max() else '#A1C9F4' for v in seasonal_df['Revenue']]
_ax2.bar(seasonal_df['MonthName'], seasonal_df['Revenue']/1e3, color=_clrs, width=0.7, edgecolor='#1D1D20')
_ax2.tick_params(axis='x', colors='#fbfbff', labelsize=10)
_ax2.tick_params(axis='y', colors='#909094', labelsize=9)
_ax2.set_ylabel('Avg Revenue (£000s)', color='#909094', fontsize=10)
_ax2.set_title('Seasonal Pattern — Average Revenue by Month', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax2.spines.values(): _sp.set_edgecolor('#444')
_ax2.grid(axis='y', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [15]:
# ── PLOT 3: YoY Monthly Revenue (2010 vs 2011) ───────────────────────────────
fig_yoy, _ax3 = plt.subplots(figsize=(9, 4.5))
fig_yoy.patch.set_facecolor('#1D1D20')
_ax3.set_facecolor('#1D1D20')
_months = yoy_pivot.index.tolist()
_xpos   = np.arange(len(_months))
_w      = 0.35
_years  = [c for c in yoy_pivot.columns if c in [2010, 2011]]
for i, yr in enumerate(_years):
    _ax3.bar(_xpos + i*_w, yoy_pivot[yr]/1e3, _w, label=str(yr),
             color=['#A1C9F4','#FFB482'][i], edgecolor='#1D1D20')
_ax3.set_xticks(_xpos + _w/2)
_ax3.set_xticklabels(_months, color='#fbfbff', fontsize=9)
_ax3.tick_params(axis='y', colors='#909094', labelsize=9)
_ax3.set_ylabel('Revenue (£000s)', color='#909094', fontsize=10)
_ax3.set_title('Year-over-Year Revenue Comparison (2010 vs 2011)', color='#fbfbff', fontsize=13, pad=12)
_ax3.legend(facecolor='#2a2a2e', labelcolor='#fbfbff', fontsize=10, framealpha=0.8)
for _sp in _ax3.spines.values(): _sp.set_edgecolor('#444')
_ax3.grid(axis='y', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [16]:
print(f"\nTop 5 months by revenue:")
print(monthly_revenue_df.nlargest(5,'Revenue')[['YearMonth','Revenue','Orders']].to_string(index=False))
print(f"\nSeasonal peak month: {seasonal_df.loc[seasonal_df['Revenue'].idxmax(),'MonthName']}")


Top 5 months by revenue:
YearMonth     Revenue  Orders
  2010-11 1166460.022    2587
  2011-11 1156205.610    2657
  2011-10 1035642.450    1929
  2010-10 1033112.010    2133
  2011-09  950690.202    1755

Seasonal peak month: Nov


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [18]:
# ── Reference date: last day in the dataset ───────────────────────────────────
_ref_date = retail_df['InvoiceDate'].max()

In [19]:
# ── RFM base ──────────────────────────────────────────────────────────────────
_cust = retail_df.groupby('CustomerID').agg(
    LastPurchase=('InvoiceDate', 'max'),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

_cust['Recency'] = (_ref_date - _cust['LastPurchase']).dt.days

In [20]:
# ── RFM Scores (1–5 quintiles) ────────────────────────────────────────────────
_cust['R_Score'] = pd.qcut(_cust['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
_cust['F_Score'] = pd.qcut(_cust['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
_cust['M_Score'] = pd.qcut(_cust['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
_cust['RFM_Score'] = _cust['R_Score'] + _cust['F_Score'] + _cust['M_Score']


In [21]:
# ── Segment mapping ───────────────────────────────────────────────────────────
def _segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    rfm = row['RFM_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r >= 3 and m >= 4:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r == 1 and f >= 2:
        return 'Lost'
    elif rfm >= 9:
        return 'Promising'
    else:
        return 'Needs Attention'

_cust['Segment'] = _cust.apply(_segment, axis=1)

In [22]:
# ── Dormant flag: no purchase in last 90 days of data ────────────────────────
_cust['IsDormant'] = _cust['Recency'] > 90

In [23]:
# ── Pareto — top 20% customers ────────────────────────────────────────────────
_cust_sorted = _cust.sort_values('Monetary', ascending=False).reset_index(drop=True)
_cust_sorted['CumRevPct'] = _cust_sorted['Monetary'].cumsum() / _cust_sorted['Monetary'].sum() * 100
_top20_cutoff = int(len(_cust_sorted) * 0.20)
_top20_rev_pct = _cust_sorted.loc[_top20_cutoff - 1, 'CumRevPct']

In [24]:
# ── Customer lifetime value (simplified: Monetary = CLV proxy) ────────────────
rfm_df = _cust.copy()

print("=== CUSTOMER ANALYTICS ===")
print(f"Total customers analysed : {len(rfm_df):,}")
print(f"Dormant (>90 days)       : {rfm_df['IsDormant'].sum():,} ({rfm_df['IsDormant'].mean()*100:.1f}%)")
print(f"Top 20% customers drive  : {_top20_rev_pct:.1f}% of revenue")
print(f"\nSegment distribution:\n{rfm_df['Segment'].value_counts().to_string()}")
print(f"\nTop 10 customers by revenue:\n{rfm_df.nlargest(10,'Monetary')[['CustomerID','Recency','Frequency','Monetary','Segment']].to_string(index=False)}")

=== CUSTOMER ANALYTICS ===
Total customers analysed : 5,878
Dormant (>90 days)       : 2,985 (50.8%)
Top 20% customers drive  : 77.2% of revenue

Segment distribution:
Segment
Needs Attention        1716
Champions              1297
Loyal Customers        1138
At Risk                 616
Lost                    465
Recent Customers        443
Promising               173
Potential Loyalists      30

Top 10 customers by revenue:
CustomerID  Recency  Frequency  Monetary          Segment
     18102        0        145 580987.04        Champions
     14646        1        151 528602.52        Champions
     14156        9        156 313437.62        Champions
     14911        0        398 291420.81        Champions
     17450        7         51 244784.25        Champions
     13694        3        143 195640.69        Champions
     17511        2         60 172132.87        Champions
     16446        0          2 168472.50 Recent Customers
     16684        3         55 147142.77        

In [25]:
# ── CHART 1: Customer Segments ────────────────────────────────────────────────
_seg_rev = rfm_df.groupby('Segment')['Monetary'].sum().sort_values(ascending=True)
_palette = ['#A1C9F4','#FFB482','#8DE5A1','#FF9F9B','#D0BBFF','#ffd400','#F7B6D2','#C49C94']

fig_segments, _ax_seg = plt.subplots(figsize=(9, 5))
fig_segments.patch.set_facecolor('#1D1D20')
_ax_seg.set_facecolor('#1D1D20')

_seg_colors = _palette[:len(_seg_rev)]
_seg_bars = _ax_seg.barh(_seg_rev.index, _seg_rev.values/1e3, color=_seg_colors, edgecolor='#1D1D20')
for _b, _v in zip(_seg_bars, _seg_rev.values/1e3):
    _ax_seg.text(_v + 5, _b.get_y() + _b.get_height()/2,
            f'£{_v:,.0f}k', va='center', color='#fbfbff', fontsize=9)
_ax_seg.tick_params(axis='y', colors='#fbfbff', labelsize=10)
_ax_seg.tick_params(axis='x', colors='#909094', labelsize=9)
_ax_seg.set_xlabel('Revenue (£000s)', color='#909094', fontsize=10)
_ax_seg.set_title('Revenue by Customer Segment (RFM)', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax_seg.spines.values():
    _sp.set_edgecolor('#444')
_ax_seg.grid(axis='x', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [26]:
# ── CHART 2: Pareto (Revenue Concentration) ──────────────────────────────────
fig_pareto, _ax_pareto = plt.subplots(figsize=(9, 5))
fig_pareto.patch.set_facecolor('#1D1D20')
_ax_pareto.set_facecolor('#1D1D20')

_n = len(_cust_sorted)
_pct_customers = np.arange(1, _n + 1) / _n * 100
_ax_pareto.plot(_pct_customers, _cust_sorted['CumRevPct'], color='#A1C9F4', linewidth=2.2)
_ax_pareto.axvline(20, color='#ffd400', linestyle='--', linewidth=1.4, label=f'Top 20% → {_top20_rev_pct:.0f}% revenue')
_ax_pareto.axhline(80, color='#FF9F9B', linestyle='--', linewidth=1.4, label='80% revenue line')
_ax_pareto.fill_between(_pct_customers, _cust_sorted['CumRevPct'], alpha=0.12, color='#A1C9F4')
_ax_pareto.set_xlabel('% of Customers (ranked by revenue)', color='#909094', fontsize=10)
_ax_pareto.set_ylabel('Cumulative % of Revenue', color='#909094', fontsize=10)
_ax_pareto.set_title('Pareto Chart — Revenue Concentration', color='#fbfbff', fontsize=13, pad=12)
_ax_pareto.tick_params(colors='#909094')
_ax_pareto.legend(facecolor='#2a2a2e', labelcolor='#fbfbff', fontsize=10)
for _sp in _ax_pareto.spines.values():
    _sp.set_edgecolor('#444')
_ax_pareto.grid(color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [27]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

BG   = '#1D1D20'
TXT  = '#fbfbff'
GREY = '#909094'
BLUE = '#A1C9F4'
ORNG = '#FFB482'
GRN  = '#8DE5A1'
GOLD = '#ffd400'
EDGE = '#333333'


In [28]:

# ── Pre-compute ───────────────────────────────────────────────────────────────
_total_rev = retail_df['Revenue'].sum()
_total_ord = retail_df['InvoiceNo'].nunique()
_total_cus = retail_df['CustomerID'].nunique()
_aov       = retail_df.groupby('InvoiceNo')['Revenue'].sum().mean()

_monthly = (retail_df.groupby('YearMonth')['Revenue'].sum()
            .reset_index().sort_values('YearMonth'))

_geo = (retail_df[retail_df['Country'] != 'United Kingdom']
        .groupby('Country')['Revenue'].sum()
        .nlargest(10).sort_values())

_prod_top = (retail_df.groupby('Description')['Revenue'].sum()
             .nlargest(10).sort_values())
_prod_top.index = _prod_top.index.str[:28].fillna('(Unknown)')

_seg_rev = rfm_df.groupby('Segment')['Monetary'].sum().sort_values()


In [29]:
# ── Figure ────────────────────────────────────────────────────────────────────
fig_dashboard = plt.figure(figsize=(18, 14), facecolor=BG)
_gs = gridspec.GridSpec(4, 4, figure=fig_dashboard,
                        hspace=0.55, wspace=0.35,
                        left=0.07, right=0.97, top=0.94, bottom=0.06)

<Figure size 1800x1400 with 0 Axes>

In [30]:
# Title row
_ax_t = fig_dashboard.add_subplot(_gs[0, :])
_ax_t.set_facecolor(BG); _ax_t.axis('off')
_ax_t.text(0.5, 0.75, 'Online Retail II — Executive Dashboard',
           ha='center', va='center', fontsize=20, fontweight='bold',
           color=TXT, transform=_ax_t.transAxes)
_ax_t.text(0.5, 0.2, 'Dec 2009 – Dec 2011  |  41 Countries  |  5,878 Customers  |  36,969 Orders',
           ha='center', va='center', fontsize=12, color=GREY, transform=_ax_t.transAxes)

Text(0.5, 0.2, 'Dec 2009 – Dec 2011  |  41 Countries  |  5,878 Customers  |  36,969 Orders')

In [31]:
# KPI cards
_kpi_data = [
    ('Total Revenue',    f'£{_total_rev/1e6:.2f}M', GOLD),
    ('Total Orders',     f'{_total_ord:,}',          BLUE),
    ('Customers',        f'{_total_cus:,}',          GRN),
    ('Avg Order Value',  f'£{_aov:,.0f}',            ORNG),
]
for _ki, (_label, _val, _clr) in enumerate(_kpi_data):
    _axk = fig_dashboard.add_subplot(_gs[1, _ki])
    _axk.set_facecolor('#25252A'); _axk.axis('off')
    _axk.text(0.5, 0.62, _val,   ha='center', va='center', fontsize=22, fontweight='bold',
              color=_clr, transform=_axk.transAxes)
    _axk.text(0.5, 0.22, _label, ha='center', va='center', fontsize=11, color=GREY, transform=_axk.transAxes)
    for _s in ['bottom','top','left','right']:
        _axk.spines[_s].set_visible(True); _axk.spines[_s].set_edgecolor('#3a3a40')

In [32]:
# Monthly trend
_ax_trend = fig_dashboard.add_subplot(_gs[2, :2])
_ax_trend.set_facecolor(BG)
_xm = range(len(_monthly))
_ax_trend.fill_between(_xm, _monthly['Revenue']/1e3, alpha=0.2, color=BLUE)
_ax_trend.plot(_xm, _monthly['Revenue']/1e3, color=BLUE, linewidth=2, marker='o', markersize=3)
_step = 4
_xt = list(range(0, len(_monthly), _step))
_ax_trend.set_xticks(_xt)
_ax_trend.set_xticklabels([_monthly['YearMonth'].iloc[_j] for _j in _xt],
                           rotation=45, ha='right', fontsize=8, color=GREY)
_ax_trend.tick_params(axis='y', colors=GREY, labelsize=8)
_ax_trend.set_title('Monthly Revenue (£000s)', color=TXT, fontsize=11, pad=8)
for _sp in _ax_trend.spines.values(): _sp.set_edgecolor(EDGE)
_ax_trend.grid(axis='y', color=EDGE, linewidth=0.5)

In [33]:
# Top countries
_ax_geo = fig_dashboard.add_subplot(_gs[2, 2:])
_ax_geo.set_facecolor(BG)
_ax_geo.barh(_geo.index, _geo.values/1e3, color=BLUE, edgecolor=BG)
_ax_geo.tick_params(axis='y', colors=TXT, labelsize=8)
_ax_geo.tick_params(axis='x', colors=GREY, labelsize=8)
_ax_geo.set_title('Top 10 Markets excl. UK (£000s)', color=TXT, fontsize=11, pad=8)
for _sp in _ax_geo.spines.values(): _sp.set_edgecolor(EDGE)
_ax_geo.grid(axis='x', color=EDGE, linewidth=0.5)

In [34]:
# Top products
_ax_prod = fig_dashboard.add_subplot(_gs[3, :2])
_ax_prod.set_facecolor(BG)
_pc = [GOLD if _pi == len(_prod_top)-1 else BLUE for _pi in range(len(_prod_top))]
_ax_prod.barh(_prod_top.index, _prod_top.values/1e3, color=_pc, edgecolor=BG)
_ax_prod.tick_params(axis='y', colors=TXT, labelsize=7.5)
_ax_prod.tick_params(axis='x', colors=GREY, labelsize=8)
_ax_prod.set_title('Top 10 Products by Revenue (£000s)', color=TXT, fontsize=11, pad=8)
for _sp in _ax_prod.spines.values(): _sp.set_edgecolor(EDGE)
_ax_prod.grid(axis='x', color=EDGE, linewidth=0.5)

In [35]:
# Customer segments
_ax_seg = fig_dashboard.add_subplot(_gs[3, 2:])
_ax_seg.set_facecolor(BG)
_seg_pal = ['#A1C9F4','#FFB482','#8DE5A1','#FF9F9B','#D0BBFF','#ffd400','#F7B6D2','#C49C94']
_ax_seg.barh(_seg_rev.index, _seg_rev.values/1e3, color=_seg_pal[:len(_seg_rev)], edgecolor=BG)
_ax_seg.tick_params(axis='y', colors=TXT, labelsize=8)
_ax_seg.tick_params(axis='x', colors=GREY, labelsize=8)
_ax_seg.set_title('Revenue by Customer Segment (£000s)', color=TXT, fontsize=11, pad=8)
for _sp in _ax_seg.spines.values(): _sp.set_edgecolor(EDGE)
_ax_seg.grid(axis='x', color=EDGE, linewidth=0.5)

plt.close('all')
print(f"Dashboard built. Revenue=£{_total_rev/1e6:.2f}M | Orders={_total_ord:,} | Customers={_total_cus:,} | AOV=£{_aov:,.0f}")

Dashboard built. Revenue=£17.37M | Orders=36,969 | Customers=5,878 | AOV=£470


In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [37]:
# ── Product summary ───────────────────────────────────────────────────────────
_prod = retail_df.groupby(['StockCode','Description']).agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalQty=('Quantity', 'sum'),
    NumOrders=('InvoiceNo', 'nunique'),
    NumCustomers=('CustomerID', 'nunique'),
    AvgUnitPrice=('UnitPrice', 'mean')
).reset_index()

In [38]:


# Repeat purchase rate: orders / customers (avg times a customer reorders)
_prod['RepeatRate'] = (_prod['NumOrders'] / _prod['NumCustomers']).round(2)

product_summary_df = _prod.sort_values('TotalRevenue', ascending=False).reset_index(drop=True)
product_summary_df['RevenueRank'] = product_summary_df['TotalRevenue'].rank(ascending=False).astype(int)

In [39]:
# ── Top 20 by revenue ─────────────────────────────────────────────────────────
_top20 = product_summary_df.head(20).copy()
_top20['Label'] = _top20['Description'].str[:35].fillna(_top20['StockCode'])

print("=== PRODUCT ANALYTICS ===")
print(f"Total products: {len(product_summary_df):,}")
print(f"\nTop 10 by revenue:")
print(product_summary_df[['StockCode','Description','TotalRevenue','NumOrders','RepeatRate']].head(10).to_string(index=False))

=== PRODUCT ANALYTICS ===
Total products: 5,315

Top 10 by revenue:
StockCode                        Description  TotalRevenue  NumOrders  RepeatRate
    22423           REGENCY CAKESTAND 3 TIER     277656.25       3317        2.52
   85123A WHITE HANGING HEART T-LIGHT HOLDER     247048.01       4888        3.28
    23843        PAPER CRAFT , LITTLE BIRDIE     168469.60          1        1.00
        M                             Manual     151777.67        620        1.42
   85099B            JUMBO BAG RED RETROSPOT     134307.44       2612        3.04
     POST                            POSTAGE     124648.04       1803        4.45
    84879      ASSORTED COLOUR BIRD ORNAMENT     124351.86       2652        2.63
    47566                      PARTY BUNTING     103283.38       2077        2.32
    23166     MEDIUM CERAMIC TOP STORAGE JAR      81416.73        195        1.41
    22086    PAPER CHAIN KIT 50'S CHRISTMAS       76598.18       1691        1.89


In [40]:
# ── Bottom / rarely sold (bottom 10% by orders) ───────────────────────────────
_bottom = product_summary_df[product_summary_df['NumOrders'] <= 2].sort_values('TotalRevenue')
print(f"\nRarely-sold products (≤2 orders): {len(_bottom):,}  ({len(_bottom)/len(product_summary_df)*100:.1f}%)")


Rarely-sold products (≤2 orders): 374  (7.0%)


In [41]:
# ── High repeat-purchase products (top 20) ────────────────────────────────────
_repeat = product_summary_df[product_summary_df['NumCustomers'] >= 10].nlargest(10, 'RepeatRate')
print(f"\nTop 10 products by repeat purchase rate:\n{_repeat[['Description','RepeatRate','NumOrders','NumCustomers']].to_string(index=False)}")


Top 10 products by repeat purchase rate:
                        Description  RepeatRate  NumOrders  NumCustomers
                           CARRIAGE        5.51        248            45
                            POSTAGE        4.45       1803           405
          WHITE RETRODISC LAMPSHADE        3.73         41            11
 WHITE HANGING HEART T-LIGHT HOLDER        3.28       4888          1490
       RECORD FRAME 7" SINGLE SIZE         3.16        319           101
             JUMBO STORAGE BAG SUKI        3.14       1714           546
  WOOD S/3 CABINET ANT WHITE FINISH        3.13       1002           320
ASSORTED COLOUR LIZARD SUCTION HOOK        3.13         72            23
            JUMBO BAG RED RETROSPOT        3.04       2612           860
                      CHILLI LIGHTS        3.03        922           304


In [42]:
# ── CHART: Top 20 Products by Revenue ────────────────────────────────────────
fig_products, _ax_prod = plt.subplots(figsize=(11, 7))
fig_products.patch.set_facecolor('#1D1D20')
_ax_prod.set_facecolor('#1D1D20')

_vals = _top20['TotalRevenue'].values / 1e3
_labels = _top20['Label'].values
_colors = ['#ffd400' if i == 0 else '#A1C9F4' for i in range(len(_vals))]

_ax_prod.barh(range(len(_labels)), _vals[::-1], color=_colors[::-1], edgecolor='#1D1D20')
_ax_prod.set_yticks(range(len(_labels)))
_ax_prod.set_yticklabels(_labels[::-1], color='#fbfbff', fontsize=8.5)
_ax_prod.tick_params(axis='x', colors='#909094', labelsize=9)
_ax_prod.set_xlabel('Revenue (£000s)', color='#909094', fontsize=10)
_ax_prod.set_title('Top 20 Products by Revenue', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax_prod.spines.values():
    _sp.set_edgecolor('#444')
_ax_prod.grid(axis='x', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [43]:

# ── Price bands (per-product average price) ───────────────────────────────────
_price_bands = [0, 2, 5, 10, 25, 50, 100, float('inf')]
_band_labels  = ['<£2','£2–5','£5–10','£10–25','£25–50','£50–100','>£100']

retail_df_p = retail_df.copy()
retail_df_p['PriceBand'] = pd.cut(retail_df_p['UnitPrice'], bins=_price_bands, labels=_band_labels, right=False)

_band_summary = retail_df_p.groupby('PriceBand', observed=True).agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Lines=('Revenue','count'),
    AvgPrice=('UnitPrice','mean')
).reset_index()
_band_summary['RevPct'] = (_band_summary['Revenue'] / _band_summary['Revenue'].sum() * 100).round(1)

print("=== PRICING ANALYSIS ===")
print(f"\nPrice Band Revenue Contribution:")
print(_band_summary[['PriceBand','Revenue','Orders','RevPct','AvgPrice']].to_string(index=False))

=== PRICING ANALYSIS ===

Price Band Revenue Contribution:
PriceBand     Revenue  Orders  RevPct   AvgPrice
      <£2 7251994.758   32881    41.7   1.130384
     £2–5 6369389.470   31710    36.7   3.236095
    £5–10 2423476.690   23638    13.9   7.512129
   £10–25  934245.110   12374     5.4  13.835636
   £25–50  106758.200    1085     0.6  35.857163
  £50–100   34174.630     395     0.2  59.894158
    >£100  254765.410     329     1.5 575.609194


In [44]:
# ── Product-level: ASP and frequency ─────────────────────────────────────────
_prod_price = retail_df_p.groupby('StockCode').agg(
    ASP=('UnitPrice','mean'),
    Orders=('InvoiceNo','nunique')
).reset_index()

_corr = _prod_price[['ASP','Orders']].corr().iloc[0,1]
print(f"\nCorrelation between Avg Selling Price and Purchase Frequency: {_corr:.3f}")
print(f"\nHighest ASP products (with ≥5 orders):")
print(_prod_price[_prod_price['Orders']>=5].nlargest(10,'ASP')[['StockCode','ASP','Orders']].to_string(index=False))


Correlation between Avg Selling Price and Purchase Frequency: -0.032

Highest ASP products (with ≥5 orders):
StockCode        ASP  Orders
      DOT 744.147500      16
    22656 214.864865      37
        M 214.785888     620
    22655 184.230769      52
    22827 158.714286      35
    22828 154.090909      11
    22823 117.500000      18
    21760 116.923077      12
    22826 114.024390      41
   ADJUST 110.578750      32


In [45]:
# ── CHART 1: Revenue by Price Band ────────────────────────────────────────────
fig_pricing, _ax_pricing = plt.subplots(figsize=(9, 5))
fig_pricing.patch.set_facecolor('#1D1D20')
_ax_pricing.set_facecolor('#1D1D20')

_clr = ['#ffd400' if v == _band_summary['Revenue'].max() else '#A1C9F4'
        for v in _band_summary['Revenue']]
_bars = _ax_pricing.bar(_band_summary['PriceBand'].astype(str), _band_summary['Revenue']/1e3,
               color=_clr, edgecolor='#1D1D20', width=0.7)
for _bar, _pct in zip(_bars, _band_summary['RevPct']):
    _ax_pricing.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 30,
            f'{_pct:.0f}%', ha='center', color='#fbfbff', fontsize=9)
_ax_pricing.tick_params(axis='x', colors='#fbfbff', labelsize=10)
_ax_pricing.tick_params(axis='y', colors='#909094', labelsize=9)
_ax_pricing.set_xlabel('Price Band (Unit Price)', color='#909094', fontsize=10)
_ax_pricing.set_ylabel('Revenue (£000s)', color='#909094', fontsize=10)
_ax_pricing.set_title('Revenue Contribution by Price Band', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax_pricing.spines.values():
    _sp.set_edgecolor('#444')
_ax_pricing.grid(axis='y', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [46]:
# ── CHART 2: ASP vs Frequency scatter (log-log) ───────────────────────────────
fig_price_scatter, _ax_scatter = plt.subplots(figsize=(8, 5))
fig_price_scatter.patch.set_facecolor('#1D1D20')
_ax_scatter.set_facecolor('#1D1D20')

_s = _prod_price[(_prod_price['ASP'] > 0) & (_prod_price['Orders'] > 0)]
_ax_scatter.scatter(np.log1p(_s['ASP']), np.log1p(_s['Orders']),
            alpha=0.35, s=18, color='#A1C9F4')
_ax_scatter.tick_params(colors='#909094', labelsize=9)
_ax_scatter.set_xlabel('ln(Avg Selling Price)', color='#909094', fontsize=10)
_ax_scatter.set_ylabel('ln(Purchase Frequency)', color='#909094', fontsize=10)
_ax_scatter.set_title(f'Price vs Purchase Frequency (Pearson r = {_corr:.2f})', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax_scatter.spines.values():
    _sp.set_edgecolor('#444')
_ax_scatter.grid(color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [48]:
# ── Country-level summary ─────────────────────────────────────────────────────
country_summary_df = (
    retail_df.groupby('Country').agg(
        Revenue=('Revenue','sum'),
        Orders=('InvoiceNo','nunique'),
        Customers=('CustomerID','nunique'),
        AvgOrderValue=('Revenue','mean')
    ).reset_index()
    .sort_values('Revenue', ascending=False)
    .reset_index(drop=True)
)
country_summary_df['RevPct'] = (country_summary_df['Revenue'] / country_summary_df['Revenue'].sum() * 100).round(2)

print("=== GEOGRAPHIC ANALYSIS ===")
print(f"Markets tracked: {len(country_summary_df)}")
print(f"\nTop 15 countries by revenue:\n{country_summary_df.head(15).to_string(index=False)}")

=== GEOGRAPHIC ANALYSIS ===
Markets tracked: 41

Top 15 countries by revenue:
        Country      Revenue  Orders  Customers  AvgOrderValue  RevPct
 United Kingdom 14389234.917   33541       5350      20.544662   82.82
           EIRE   616570.540     567          5      39.612627    3.55
    Netherlands   554038.090     228         22     108.955377    3.19
        Germany   425019.711     789        107      25.865367    2.45
         France   348768.960     614         95      25.813704    2.01
      Australia   169283.460      95         15      94.624628    0.97
          Spain   108332.490     154         41      29.582875    0.62
    Switzerland   100061.940      90         22      33.298483    0.58
         Sweden    91515.820     104         19      69.488094    0.53
        Denmark    68580.690      43         12      88.149987    0.39
        Belgium    65387.820     149         29      21.403542    0.38
         Norway    56322.500      45         13      43.694725    0.32

In [49]:
# ── YoY growth by country (2010 → 2011) ──────────────────────────────────────
_yoy = retail_df.groupby(['Country','Year'])['Revenue'].sum().unstack(fill_value=0)
_yoy.columns = [int(c) for c in _yoy.columns]
if 2010 in _yoy.columns and 2011 in _yoy.columns:
    _yoy['Growth_pct'] = ((_yoy[2011] - _yoy[2010]) / _yoy[2010].replace(0, np.nan) * 100).round(1)
    _growing = _yoy[(_yoy[2010] >= 10000) & _yoy['Growth_pct'].notna()].sort_values('Growth_pct', ascending=False).head(10)
    print(f"\nFastest-growing markets (2010→2011, ≥£10k base):\n{_growing[[2010,2011,'Growth_pct']].to_string()}")


Fastest-growing markets (2010→2011, ≥£10k base):
                   2010       2011  Growth_pct
Country                                       
Australia     31523.900  137488.46       336.1
Japan         13312.610   29711.30       123.2
Belgium       25553.790   39386.43        54.1
Spain         40667.480   59714.83        46.8
Portugal      21797.330   30935.87        41.9
France       142911.270  199336.00        39.5
Norway        23458.870   32378.32        38.0
Switzerland   44333.510   55139.03        24.4
Italy         15014.080   16671.74        11.0
Germany      201716.781  213472.66         5.8


In [50]:
# ── CHART 1: Top 10 Countries by Revenue (ex UK) ─────────────────────────────
_top10_ex_uk = country_summary_df[country_summary_df['Country'] != 'United Kingdom'].head(10)

fig_geo, _ax_geo = plt.subplots(figsize=(10, 5.5))
fig_geo.patch.set_facecolor('#1D1D20')
_ax_geo.set_facecolor('#1D1D20')

_clrs = ['#ffd400' if i == 0 else '#A1C9F4' for i in range(len(_top10_ex_uk))]
_ax_geo.barh(_top10_ex_uk['Country'].values[::-1], _top10_ex_uk['Revenue'].values[::-1]/1e3,
        color=_clrs[::-1], edgecolor='#1D1D20')
_ax_geo.tick_params(axis='y', colors='#fbfbff', labelsize=10)
_ax_geo.tick_params(axis='x', colors='#909094', labelsize=9)
_ax_geo.set_xlabel('Revenue (£000s)', color='#909094', fontsize=10)
_ax_geo.set_title('Top 10 International Markets by Revenue (excl. UK)', color='#fbfbff', fontsize=13, pad=12)
for _sp in _ax_geo.spines.values():
    _sp.set_edgecolor('#444')
_ax_geo.grid(axis='x', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')

In [51]:
# ── CHART 2: UK vs International revenue split ────────────────────────────────
_uk_rev  = country_summary_df.loc[country_summary_df['Country']=='United Kingdom','Revenue'].values[0]
_int_rev = country_summary_df.loc[country_summary_df['Country']!='United Kingdom','Revenue'].sum()

fig_geo_split, _ax_geo_split = plt.subplots(figsize=(7, 5))
fig_geo_split.patch.set_facecolor('#1D1D20')
_ax_geo_split.set_facecolor('#1D1D20')

_wedges, _texts, _autotexts = _ax_geo_split.pie(
    [_uk_rev, _int_rev], labels=['United Kingdom','International'],
    autopct='%1.1f%%', colors=['#A1C9F4','#FFB482'], startangle=140,
    textprops={'color':'#fbfbff','fontsize':12},
    wedgeprops={'edgecolor':'#1D1D20','linewidth':2}
)
for _at in _autotexts:
    _at.set_color('#1D1D20'); _at.set_fontsize(11)
_ax_geo_split.set_title('Revenue: UK vs International', color='#fbfbff', fontsize=13, pad=12)
plt.tight_layout()
plt.close('all')

print(f"\nUK revenue:            £{_uk_rev:,.0f} ({_uk_rev/(_uk_rev+_int_rev)*100:.1f}%)")
print(f"International revenue: £{_int_rev:,.0f} ({_int_rev/(_uk_rev+_int_rev)*100:.1f}%)")


UK revenue:            £14,389,235 (82.8%)
International revenue: £2,985,569 (17.2%)


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [53]:
# ── Invoice-level aggregation ─────────────────────────────────────────────────
invoice_df = retail_df.groupby(['InvoiceNo','CustomerID','Country','InvoiceDate']).agg(
    InvoiceRevenue=('Revenue','sum'),
    InvoiceQty=('Quantity','sum'),
    LineItems=('StockCode','nunique')
).reset_index()

In [54]:
# ── Outlier detection: IQR method on InvoiceRevenue ──────────────────────────
_q1  = invoice_df['InvoiceRevenue'].quantile(0.25)
_q3  = invoice_df['InvoiceRevenue'].quantile(0.75)
_iqr = _q3 - _q1
_upper = _q3 + 3 * _iqr

invoice_df['IsLargeOrder'] = invoice_df['InvoiceRevenue'] > _upper
_large = invoice_df[invoice_df['IsLargeOrder']].sort_values('InvoiceRevenue', ascending=False)

print("=== OPERATIONAL INSIGHTS ===")
print(f"\nIQR upper fence (3×IQR): £{_upper:,.2f}")
print(f"Unusually large invoices: {len(_large):,} ({len(_large)/len(invoice_df)*100:.2f}% of all orders)")
print(f"\nTop 10 largest invoices:")
print(_large[['InvoiceNo','CustomerID','Country','InvoiceDate','InvoiceRevenue','InvoiceQty','LineItems']].head(10).to_string(index=False))

=== OPERATIONAL INSIGHTS ===

IQR upper fence (3×IQR): £1,434.79
Unusually large invoices: 1,439 (3.89% of all orders)

Top 10 largest invoices:
InvoiceNo CustomerID        Country         InvoiceDate  InvoiceRevenue  InvoiceQty  LineItems
   581483      16446 United Kingdom 2011-12-09 09:15:00       168469.60       80995          1
   541431      12346 United Kingdom 2011-01-18 10:01:00        77183.60       74215          1
   493819      14156           EIRE 2010-01-07 12:34:00        44051.60       25018         94
   556444      15098 United Kingdom 2011-06-10 15:28:00        38970.00          60          1
   524181      17450 United Kingdom 2010-09-27 16:59:00        33167.80        8172         13
   567423      17450 United Kingdom 2011-09-20 11:05:00        31698.16       12572         12
   526934      18102 United Kingdom 2010-10-14 09:46:00        26007.08        5079         15
   515944      18102 United Kingdom 2010-07-15 15:29:00        22863.36        4992         17


In [55]:
# ── Bulk purchasers: customers with consistently high quantities ──────────────
_bulk = (
    retail_df.groupby('CustomerID').agg(
        AvgQtyPerLine=('Quantity','mean'),
        TotalQty=('Quantity','sum'),
        Orders=('InvoiceNo','nunique')
    ).reset_index()
)
_bulk_thresh = _bulk['AvgQtyPerLine'].quantile(0.95)
_bulk_buyers = _bulk[_bulk['AvgQtyPerLine'] >= _bulk_thresh].sort_values('TotalQty', ascending=False)
print(f"\nBulk purchasers (top 5% avg qty/line, threshold {_bulk_thresh:.0f} units):")
print(_bulk_buyers.head(10).to_string(index=False))


Bulk purchasers (top 5% avg qty/line, threshold 45 units):
CustomerID  AvgQtyPerLine  TotalQty  Orders
     14646      95.399584    367193     151
     13902    3501.587302    220600       5
     13694     123.897959    188201     143
     18102     174.658654    181645     145
     17511      62.726981    117174      60
     16684     145.974930    104810      55
     12415      98.754860     91447      28
     14277     236.955145     89806       9
     13687    1937.044444     87167       1
     17940    2432.800000     85148      16


In [56]:
# ── Ordering frequency anomalies: orders per day ─────────────────────────────
_daily = retail_df.groupby(retail_df['InvoiceDate'].dt.date)['InvoiceNo'].nunique()
_daily_mean = _daily.mean()
_daily_std  = _daily.std()
_spike_days = _daily[_daily > _daily_mean + 3*_daily_std]
print(f"\nDaily order avg: {_daily_mean:.1f} | Std: {_daily_std:.1f}")
print(f"Spike days (>3σ): {len(_spike_days)}")
if len(_spike_days):
    print(_spike_days.sort_values(ascending=False).head(5).to_string())


Daily order avg: 61.2 | Std: 23.8
Spike days (>3σ): 3
InvoiceDate
2010-11-25    146
2010-12-02    137
2011-11-17    136


In [57]:
# ── CHART: Invoice Revenue Distribution (log scale) ──────────────────────────
fig_ops, _ax_ops = plt.subplots(figsize=(9, 4.5))
fig_ops.patch.set_facecolor('#1D1D20')
_ax_ops.set_facecolor('#1D1D20')

_rev_log = np.log1p(invoice_df['InvoiceRevenue'])
_ax_ops.hist(_rev_log, bins=60, color='#A1C9F4', edgecolor='#1D1D20', alpha=0.85)
_ax_ops.axvline(np.log1p(_upper), color='#f04438', linewidth=2, linestyle='--',
           label=f'Outlier threshold £{_upper:,.0f}')
_ax_ops.tick_params(colors='#909094', labelsize=9)
_ax_ops.set_xlabel('ln(Invoice Revenue + 1)', color='#909094', fontsize=10)
_ax_ops.set_ylabel('Number of Invoices', color='#909094', fontsize=10)
_ax_ops.set_title('Invoice Revenue Distribution (log scale)', color='#fbfbff', fontsize=13, pad=12)
_ax_ops.legend(facecolor='#2a2a2e', labelcolor='#fbfbff', fontsize=10)
for _sp in _ax_ops.spines.values():
    _sp.set_edgecolor('#444')
_ax_ops.grid(axis='y', color='#333', linewidth=0.6)
plt.tight_layout()
plt.close('all')